# Reading GGMN observations

This notebook introduces how to use the `hydropandas` package to read, process
and visualise groundwater level data from the Global Groundwater Monitoring
Network (GGMN). GGMN is maintained by UN-IGRAC and aggregates in-situ
groundwater level measurements from monitoring networks around the world.
Data are retrieved through the public GGMN WFS and web interface.

In [ ]:
import contextily as ctx
import matplotlib.pyplot as plt

import hydropandas as hpd
from hydropandas.io import ggmn

# enabling logging so we can see what happens in the background
hpd.util.get_color_logger("INFO");

## Read GGMN observations within an extent

Use `hpd.read_ggmn` to download groundwater level observations for all GGMN
monitoring locations within a bounding box. The `extent` is
`[xmin, xmax, ymin, ymax]` in the coordinate system given by `crs`.
Use `only_metadata=True` for a fast first look without measurements.

In [ ]:
# read GGMN station metadata only (fast)
# extent: [xmin, xmax, ymin, ymax] in WGS84
extent = [5.1, 5.13, 52.0, 52.05]

oc_meta = hpd.read_ggmn(
    extent=extent,
    crs=4326,
    only_metadata=True,
    max_locations=50,
)
print(f"Found {len(oc_meta)} GGMN locations in the extent")
oc_meta

In [ ]:
# plot monitoring locations on a map
ax = oc_meta.to_gdf().plot(figsize=(8, 6), color="steelblue", markersize=60, zorder=2)
ctx.add_basemap(ax=ax, crs=4326, attribution=False)

for idx, row in oc_meta.iterrows():
    ax.annotate(
        text=idx,
        xy=(row["x"], row["y"]),
        fontsize=7,
        ha="center",
        va="bottom",
    )
ax.set_title("GGMN monitoring locations")
plt.tight_layout()

In [ ]:
# download groundwater level measurements for the same extent
oc = hpd.read_ggmn(
    extent=extent,
    crs=4326,
    tmin="2010-01-01",
    tmax="2026-12-31",
    keep_all_obs=True,
    max_locations=10,
    max_pages=10,
    timeout=120,
)
oc

## Plot observations from a single monitoring well

In [ ]:
# select the first well that actually has measurements and plot the time series
o = None
for _, row in oc.iterrows():
    if not row.obs.empty:
        o = row.obs
        break

if o is not None:
    print(f"Well: {o.name}  |  unit: {o.unit}")
    o["groundwater_level"].plot(
        figsize=(12, 4),
        marker=".",
        linewidth=0.8,
        ylabel=f"Groundwater level ({o.unit})",
        title=f"GGMN groundwater level – {o.name}",
    )
    plt.tight_layout()
else:
    print("No measurements found in the downloaded collection.")

## Retrieve measurements for a known record

If you know the GGMN record ID you can call `ggmn.get_level_measurements`
directly to get a tidy `DataFrame` without creating a full `ObsCollection`.

In [ ]:
# fetch groundwater level data for a specific GGMN record
record_id = 695507  # known record in the Netherlands

df, unit = ggmn.get_level_measurements(
    record_id=record_id,
    parameter="Water level elevation a.m.s.l.",
    max_pages=10,
    timeout=120,
)
print(f"Retrieved {len(df)} measurements  |  unit: {unit}")
df.head(10)

In [ ]:
# plot the retrieved time series
if not df.empty:
    df["groundwater_level"].plot(
        figsize=(12, 4),
        marker=".",
        linewidth=0.8,
        ylabel=f"Groundwater level ({unit})",
        title=f"GGMN record {record_id}",
    )
    plt.tight_layout()

## Filter by parameter name

Some GGMN monitoring locations report multiple parameters (e.g. water level
above sea level and depth below surface). Use the `parameter` argument to
filter for a specific parameter string.

In [ ]:
# download only 'depth below surface' measurements
oc_depth = hpd.read_ggmn(
    extent=extent,
    crs=4326,
    tmin="2010-01-01",
    tmax="2026-12-31",
    parameter="depth",
    keep_all_obs=True,
    max_locations=10,
    max_pages=10,
    timeout=120,
)
print(f"Observations with 'depth' parameter: {len(oc_depth)}")
oc_depth

In [ ]:
# interactive map of downloaded monitoring locations
oc[["lat", "lon"]] = oc[["y", "x"]]
oc.plots.interactive_map(popup_width=300)